# # 05 · Embedding: from many lipids to a handful of programs

Companion to the tutorial notebook [`05_embedding`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/tree/main/notebooks/level2).

**Kernel:** `cajal-lipidomics` · **Reads:** `03_normalized.h5ad` · **Writes:** `data/derived/05_embedded.h5ad`

Every pixel is a point in a ~100-dimensional lipid space, and nobody can see structure in 100 dimensions. This notebook compresses each pixel into a handful of interpretable **lipid programs**, following the backbone every spatial atlas rests on:

> **Moran's I** feature selection → **NMF** (the embedding) → **t-SNE** (just to look) → **Harmony** (only for clustering)

**Two rules we carry from here on**
- We **learn on the control** brain and **apply** to both, so both sections share one vocabulary.
- **Harmony output is only for clustering and label transfer, never for differential tests** (with two sections, batch and condition are the same thing, so Harmony would erase the pregnancy effect).

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------
from lipid_project import paths
paths.ensure_dirs()
paths.summary()

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

import cajal_lipidomics as cl
from cajal_lipidomics import analysis, embedding, plotting
from cajal_lipidomics.style import FS
cl.style.set_style()

SEED = 0
rng = np.random.default_rng(SEED)
N_JOBS = os.cpu_count() or 1          # use every CPU core for the heavy steps (t-SNE, neighbours)

print("ready. cajal_lipidomics", cl.__version__)

# ## Load the normalized sections

We embed the **uMAIA-normalized** values (`layers["umaia"]`), rescaled per lipid to [0, 1] with `min01_per_lipid` (notebook 03) so different lipids are comparable.

In [ ]:
adata = ad.read_h5ad(paths.stage("normalized"))    # notebook 03's output (04 didn't write a stage)
Xn = analysis.min01_per_lipid(adata.layers["umaia"])       # pixels x lipids, each lipid in [0, 1]
names = adata.var["lipid"].astype(str).to_numpy()    # readable name per column

is_ctrl = (adata.obs["Condition"] == "naive").to_numpy()    # True = control pixel
is_preg = ~is_ctrl    # ~ flips True/False: the pregnant pixels
print("pixels x lipids:", Xn.shape, "| control", is_ctrl.sum(), "| pregnant", is_preg.sum())
print(f"named lipids: {(~pd.Series(names).str.startswith('ion_')).sum()} of {len(names)}")

# We'll scatter values onto the brain a lot, so one small helper. It clips the colour scale to the 2nd to 98th percentile so a few hot pixels don't wash everything out.

In [ ]:
def spatial_map(ax, obs, values, vmin=None, vmax=None, cmap="plasma", s=2):
    """Scatter `values` at each pixel's CCF position (zccf, -yccf), brain upright."""
    values = np.asarray(values)    # accept lists, Series or arrays
    vmin = np.quantile(values, 0.02) if vmin is None else vmin    # default colour floor: 2nd percentile
    vmax = np.quantile(values, 0.98) if vmax is None else vmax    # default colour ceiling: 98th percentile
    sc = ax.scatter(obs["zccf"], -obs["yccf"], c=values, cmap=cmap, s=s, vmin=vmin, vmax=vmax, rasterized=True)
    cl.style.spatial_axes(ax)
    return sc    # returned so callers can attach a colorbar


# A myelin lipid on both sections, as a first look
# HexCer 42:2 if annotated, else the first sphingolipid, else column 0
j_my = next((i for i, n in enumerate(names) if n.startswith("HexCer 42:2")),
            next((i for i, n in enumerate(names) if n.startswith(("HexCer", "SM", "Cer"))), 0))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (lab, m) in zip(axes, [("naive", is_ctrl), ("pregnant", is_preg)]):
    spatial_map(ax, adata.obs[m], Xn[m, j_my])
    ax.set_title(f"{lab}: {names[j_my]}", fontsize=FS["m"])
plt.tight_layout()
plt.show()

# ## 1. Feature selection: keep the lipids that paint anatomy

Some columns draw crisp anatomy, others are mostly speckle. Feeding speckle into the embedding wastes programs on junk. The test: **real biology is spatially smooth** (neighbouring pixels look alike), noise isn't. One number per lipid that measures this is **Moran's I**.

Four moves:
1. **Who is a neighbour?** Each pixel's 6 nearest pixels in space (a kNN graph), weight `w_ij = 1` if neighbours.
2. **Centre** the values: `x_i − x̄`.
3. **Multiply each pixel by its neighbours** and sum: positive if they agree, negative if not.
4. **Normalise:**

$$I = \frac{N}{W}\,\frac{\sum_{i}\sum_{j} w_{ij}(x_i-\bar x)(x_j-\bar x)}{\sum_i (x_i-\bar x)^2}$$

`I ≈ 1`: strong spatial structure. `I ≈ 0`: salt-and-pepper. By hand first, on the control section.

In [ ]:
from scipy.spatial import cKDTree

coords_ctrl = adata.obs.loc[is_ctrl, ["zccf", "yccf"]].to_numpy()    # in-plane coordinates (mm) of control pixels
Xc = Xn[is_ctrl]    # control pixels x lipids (0-1 scaled)

# 1. kNN graph: query k+1 because each pixel's nearest point is itself (distance 0), then drop it
tree = cKDTree(coords_ctrl)    # a spatial search tree over the pixel positions
_, idx = tree.query(coords_ctrl, k=6 + 1)    # the 7 nearest points of every pixel (the first is itself)
idx = idx[:, 1:]                                   # (pixels, 6): row i lists pixel i's 6 neighbours


def moran_from_graph(x, idx):
    """Moran's I of values x on a fixed kNN graph `idx` (steps 2-4)."""
    xc = x - x.mean()                              # 2. centre
    denom = (xc ** 2).sum()    # total variation (the denominator)
    if denom == 0:
        return 0.0                                 # a constant column has no structure at all
    num = (xc[:, None] * xc[idx]).sum()            # 3. pixel value x each neighbour's value, summed over edges
    return (len(x) / idx.size) * (num / denom)     # 4. normalise; idx.size = W, the number of edges


moran_hand = moran_from_graph(Xc[:, j_my], idx)    # our version...
moran_helper = analysis.morans_i(Xc[:, j_my], coords_ctrl, k=6)    # ...and the tutorial helper: they should match
print(f"Moran's I of {names[j_my]}: by hand {moran_hand:.4f} | helper {moran_helper:.4f}")

# Now every column, on **both** sections separately: a feature is only useful for comparing sections if it's spatially structured in each.

In [ ]:
coords_preg = adata.obs.loc[is_preg, ["zccf", "yccf"]].to_numpy()    # same graph construction for the pregnant section
_, idx_p = cKDTree(coords_preg).query(coords_preg, k=7)
idx_p = idx_p[:, 1:]    # drop self

# one Moran's I per lipid column, per section
moran_ctrl = np.array([moran_from_graph(Xc[:, c], idx) for c in range(Xn.shape[1])])
moran_preg = np.array([moran_from_graph(Xn[is_preg][:, c], idx_p) for c in range(Xn.shape[1])])

table = pd.Series(moran_ctrl, index=[f"{n} [{i}]" for i, n in enumerate(names)]).sort_values()    # label each score "name [column]" and sort, to print the extremes
print("lowest Moran's I (noisiest):\n" + table.head(4).round(3).to_string())
print("\nhighest Moran's I (most anatomical):\n" + table.tail(4).round(3).to_string())

best, worst = int(np.argmax(moran_ctrl)), int(np.argmin(moran_ctrl))    # most and least spatially structured columns
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, c in zip(axes, [best, worst]):
    spatial_map(ax, adata.obs[is_ctrl], Xc[:, c])
    ax.set_title(f"{names[c]}\nMoran's I = {moran_ctrl[c]:.2f}", fontsize=FS["m"])
fig.suptitle("high Moran = real anatomy   vs   low Moran = noise", fontsize=FS["l"])
plt.tight_layout()
plt.show()

# ### Is the 0.4 cutoff principled? A permutation null

What does Moran's I look like when there's **genuinely no** spatial structure? We simulate that: **shuffle** a lipid's values across pixels. Same numbers, same histogram, but any spatial pattern is destroyed. Recompute I hundreds of times and we get the **null distribution**. The **empirical p-value** is the fraction of shuffles that match or beat the real value, with +1 on top and bottom so it's never exactly 0: `(1 + #{null ≥ observed}) / (1 + B)`.

We do it for the myelin lipid and for a deliberately scrambled copy of it (a negative control that *should* look like noise). This permutation idea comes back in notebooks 07 and 08.

In [ ]:
B = 300    # number of shuffles; more shuffles = a finer-grained p-value
scrambled = rng.permutation(Xc[:, j_my])                   # same values, spatial order destroyed
null, obs_I, pval = {}, {}, {}    # per tag: null draws, observed I, empirical p
for tag, x in [(f"real ({names[j_my]})", Xc[:, j_my]), ("scrambled control", scrambled)]:
    obs_I[tag] = moran_from_graph(x, idx)    # the real value
    null[tag] = np.array([moran_from_graph(rng.permutation(x), idx) for _ in range(B)])    # B shuffled values (same graph, values scrambled)
    pval[tag] = (1 + (null[tag] >= obs_I[tag]).sum()) / (1 + B)    # fraction of shuffles at least as extreme
    print(f"{tag:>24}: observed I = {obs_I[tag]:+.3f} | null mean {null[tag].mean():+.4f} "
          f"(sd {null[tag].std():.4f}) | empirical p = {pval[tag]:.4f}")

fig, ax = plt.subplots(figsize=(8, 3.6))
for (tag, col) in zip(null, ["mediumpurple", "gray"]):
    ax.hist(null[tag], bins=40, alpha=0.6, color=col, label=f"null: {tag}")
    ax.axvline(obs_I[tag], color=col, lw=2)
ax.axvline(0.4, color="k", ls="--", lw=1.5, label="0.4 threshold")
ax.set_xlabel("Moran's I"); ax.set_ylabel("shuffles"); ax.legend()
ax.set_title("what Moran's I does when there is no spatial structure", fontsize=FS["m"])
plt.tight_layout()
plt.show()

# The null hugs zero, the real lipid sits far to the right, and 0.4 lands in the empty gap between them. Most real columns beat their own null even when weak, so **0.4 is a strength threshold, not a significance threshold**: it keeps only columns strong enough to anchor regional programs.

Apply it: keep columns above 0.4 **in both sections**.

In [ ]:
MORAN_THR = 0.4    # the atlas's strength cutoff
keep = (moran_ctrl > MORAN_THR) & (moran_preg > MORAN_THR)    # structured in BOTH sections
sel = np.where(keep)[0]    # column numbers we keep
sel_names = names[sel]    # their names
print(f"kept {keep.sum()} of {len(keep)} columns (Moran's I > {MORAN_THR} in both sections)")
print("dropped named lipids:", sorted({n for n in names[~keep] if not n.startswith("ion_")}))

Xc_sel, Xn_sel = Xc[:, sel], Xn[:, sel]                   # control-only and all-pixel matrices, selected columns

# ## Molecular modules: which lipids travel together

Compression only works because lipids don't move independently: myelin lipids rise wherever myelin is. A **lipid-lipid correlation heatmap** (clustered so co-varying lipids sit together) shows these **modules** as red blocks on the diagonal. These blocks are roughly the programs NMF is about to find.

In [ ]:
uniq = ad.utils.make_index_unique(pd.Index(sel_names))      # a few names repeat (two adducts), so suffix them
panel = ad.AnnData(Xc_sel.astype("float32"), var=pd.DataFrame(index=uniq))    # a small AnnData so the helper can label rows/columns
plotting.lipid_lipid_corr(panel, figsize=(10, 9))
plt.show()

C = np.corrcoef(Xc_sel.T)    # lipid x lipid correlation (corrcoef wants variables as rows, hence .T)
np.fill_diagonal(C, -9)    # a lipid with itself (r = 1) shouldn't win
C[sel_names[:, None] == sel_names[None, :]] = -9            # ignore pairs that are the same lipid twice
a, b = np.unravel_index(np.argmax(C), C.shape)    # row/column of the largest remaining correlation
print(f"most correlated pair of distinct lipids: {sel_names[a]} & {sel_names[b]} (r = {C[a, b]:.2f})")

# ## 2. NMF: compress into additive programs

Non-negative matrix factorisation approximates our data matrix `V` (pixels x lipids) as `V ≈ W H`:
- `H` (programs x lipids): each row is a **recipe**, how much of each lipid belongs to a program
- `W` (pixels x programs): each row says **how much of each program** a pixel expresses

Everything is **non-negative**, so a pixel is literally a *sum of parts*. That's what makes programs readable as biology.

**NMF vs PCA.** PCA finds directions of maximum variance, but its components mix lipids with + and − signs ("more PC minus SM"), and negative lipid amounts mean nothing. NMF's parts are purely additive. Toy example, 4 pixels x 3 lipids:

In [ ]:
from sklearn.decomposition import NMF, PCA

V = np.array([[0.1, 0.2, 9.0],      # pixels 0-1: "myelin-like", high in lipid C
              [0.2, 0.1, 8.0],
              [7.0, 6.0, 0.3],      # pixels 2-3: "grey-like", high in lipids A and B
              [8.0, 7.0, 0.2]])
toy = NMF(n_components=2, init="nndsvda", random_state=SEED, max_iter=500)    # 2 programs; nndsvda = a deterministic, sensible starting point
W_toy = toy.fit_transform(V)    # fit, and return W (pixels x programs)
H_toy = toy.components_    # H (programs x lipids)
print("W (pixels x programs):\n", W_toy.round(2))
print("H (programs x lipids), the two recipes:\n", H_toy.round(2))
print("W @ H rebuilds V:\n", (W_toy @ H_toy).round(2))

pca = PCA(n_components=2, random_state=SEED).fit(V)    # PCA on the same toy, for comparison
print("\nPCA components (can be negative):\n", pca.components_.round(2))
print("any NMF weight < 0?", bool((H_toy < 0).any()), "| any PCA loading < 0?", bool((pca.components_ < 0).any()))

# ### Seeded NMF on the control brain

Plain NMF starts from a generic guess and can drift between runs. The atlas uses a **seeded** start, implemented in `embedding.seeded_nmf`:
1. distance between lipids = `1 − |correlation|`
2. cluster lipids into `n_factors` groups by that distance
3. take each group's most central lipid as a seed program
4. run NMF starting from those seeds

We ask for 12 programs (or fewer if few columns survived selection, since you can't have more programs than features). The `ConvergenceWarning` you may see is harmless: NMF keeps polishing by tiny amounts after it has settled.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)    # hide the harmless max_iter notice

N_FACTORS = int(min(12, max(2, len(sel) // 2)))             # 12 on the real data
W_ctrl, H, nmf_model = embedding.seeded_nmf(Xc_sel, n_factors=N_FACTORS, random_state=SEED, max_iter=400)
N_FACTORS = W_ctrl.shape[1]          # read back the real count (an empty lipid cluster yields one seed fewer)
print("W (control pixels x programs):", W_ctrl.shape, "| H (programs x lipids):", H.shape)
print("all non-negative:", bool((W_ctrl >= 0).all() and (H >= 0).all()))

# Each program has two faces: its **recipe** (a row of H, the top lipids) and its **map** (a column of W painted on the brain). A program is useful when the two agree. We show the three programs with the most spatial variation.

In [ ]:
prog_order = np.argsort(W_ctrl.var(axis=0))[::-1]           # most spatially variable programs first
fig, axes = plt.subplots(3, 2, figsize=(11, 11), gridspec_kw={"width_ratios": [1.1, 1.0]})
for r, p in enumerate(prog_order[:3]):    # one row of panels per program
    top = np.argsort(H[p])[::-1][: min(8, H.shape[1])]       # this program's highest-weighted lipids
    axes[r, 0].barh(range(len(top)), H[p][top][::-1], color="mediumpurple")
    axes[r, 0].set_yticks(range(len(top))); axes[r, 0].set_yticklabels(sel_names[top][::-1], fontsize=FS["xs"])
    axes[r, 0].set_title(f"program {p}: recipe (row of H)", fontsize=FS["m"])
    spatial_map(axes[r, 1], adata.obs[is_ctrl], W_ctrl[:, p], vmin=0)
    axes[r, 1].set_title(f"program {p}: activity per pixel (column of W)", fontsize=FS["m"])
plt.tight_layout()
plt.show()

# ### Apply the control-learned programs to both sections

Keep `H` fixed and solve for each pixel's activities `W`. That's `model.transform`, wrapped by `embedding.apply_nmf`. Every pixel of both sections now speaks the same 12-program vocabulary. Sanity check: on control pixels, applying the model should reproduce the `W` we learned.

In [ ]:
W_all = embedding.apply_nmf(nmf_model, Xn_sel)    # H fixed, solve W for every pixel of both sections
adata.obsm["X_nmf"] = W_all.astype("float32")    # store per-pixel program activities
# learned vs re-applied, program by program
corr = [np.corrcoef(W_ctrl[:, k], W_all[is_ctrl][:, k])[0, 1] for k in range(N_FACTORS)]
print("all-pixel embedding:", W_all.shape, f"| min learned-vs-applied correlation: {min(corr):.3f}")

# ## 3. t-SNE: a flat picture, for looking only

t-SNE squashes the 12 program dimensions into 2, keeping each pixel's **near neighbours** near. We always run it on the embedding, never on raw lipids: faster, denoised, and raw high-dimensional distances are unreliable.

> **For looking, not measuring.** Distances between blobs, blob sizes and axes mean nothing. Use it to spot groups, then go back to the numbers.

This takes a few minutes on the full data.

In [ ]:
from openTSNE import TSNE

X_tsne = np.asarray(TSNE(n_components=2, perplexity=30, n_jobs=N_JOBS, random_state=SEED).fit(W_all))    # 2D layout of the program space
adata.obsm["X_tsne"] = X_tsne.astype("float32")    # stored for later notebooks

order = rng.permutation(adata.n_obs)            # random draw order, so neither section sits on top
ts, obs_o = X_tsne[order], adata.obs.iloc[order]    # coordinates and metadata in the shuffled draw order
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
for ax, key, title in [(axes[0], "SectionID", "by section (the batch)"), (axes[1], "Condition", "by condition")]:
    for val, colour in zip(sorted(obs_o[key].unique()), ["tab:orange", "tab:purple"]):
        m = (obs_o[key] == val).to_numpy()
        ax.scatter(ts[m, 0], ts[m, 1], s=2, alpha=0.5, color=colour, label=str(val), rasterized=True)
    ax.set_title(title); ax.legend(markerscale=4)
axes[2].scatter(ts[:, 0], ts[:, 1], s=2, c=obs_o["allencolor"], alpha=0.6, rasterized=True)
axes[2].set_title("by Allen region (anatomy)")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

# Islands follow Allen regions: the embedding separates anatomy. But inside islands the two sections often sit side by side instead of interleaving. That residual split is the **batch effect**. (By section and by condition look identical because, with two sections, they *are* the same thing.)

## 4. Harmony: make the sections overlap, for clustering only

**Harmony** works on the embedding coordinates. It repeats two moves: softly cluster all pixels ignoring batch, then within each soft cluster nudge each batch's pixels toward the shared centre. Biologically identical groups melt together; different groups stay apart. It's much more aggressive than uMAIA.

To put a number on "mixing" we use a simple score: for a random sample of pixels, what fraction of their 30 nearest neighbours in the embedding come from the **other** section? Perfect mixing would match the other section's share of all pixels; no mixing gives ~0.

In [ ]:
from sklearn.neighbors import NearestNeighbors


def mixing_score(Z, batch, n=5000, k=30, seed=SEED):
    """Mean fraction of each sampled pixel's k nearest neighbours that come from a different batch."""
    r = np.random.default_rng(seed)
    pick = r.choice(len(Z), size=min(n, len(Z)), replace=False)    # a random sample of pixels to score
    nn = NearestNeighbors(n_neighbors=k + 1, n_jobs=N_JOBS).fit(Z)    # neighbour search over ALL pixels
    _, nb = nn.kneighbors(Z[pick])    # each sampled pixel's k+1 nearest (incl. itself)
    nb = nb[:, 1:]                                   # drop the pixel itself
    return float((batch[nb] != batch[pick][:, None]).mean())    # share of neighbours from a different section


batch = adata.obs["SectionID"].astype(str).to_numpy()    # the batch label = which section
Z_harm = embedding.harmonize(W_all, batch, random_state=SEED)    # Harmony on the program space, batch = section
adata.obsm["X_harmony"] = Z_harm.astype("float32")    # for clustering / transfer ONLY

ideal = 1 - max(is_ctrl.mean(), is_preg.mean())       # rough target if sections mixed perfectly
print(f"cross-section neighbour fraction: before Harmony {mixing_score(W_all, batch):.3f} | "
      f"after {mixing_score(Z_harm, batch):.3f} | perfect mixing ~ {ideal:.2f}")

In [ ]:
ts_h = np.asarray(TSNE(n_components=2, perplexity=30, n_jobs=N_JOBS, random_state=SEED).fit(Z_harm))[order]    # a second t-SNE, on the harmonized space, same draw order
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, coords2d, title in [(axes[0], ts, "before Harmony"), (axes[1], ts_h, "after Harmony")]:
    for val, colour in zip(sorted(obs_o["SectionID"].unique()), ["tab:orange", "tab:purple"]):
        m = (obs_o["SectionID"] == val).to_numpy()
        ax.scatter(coords2d[m, 0], coords2d[m, 1], s=2, alpha=0.5, color=colour, label=f"section {val:.0f}",
                   rasterized=True)
    ax.set_title(f"{title}: coloured by section", fontsize=FS["m"]); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(markerscale=4)
plt.tight_layout()
plt.show()

# > **Read this twice.** `X_harmony` is for **clustering and label transfer only**. Differential tests ("which lipids changed in pregnancy?") always go back to the untouched uMAIA values. With two sections, condition and batch are confounded, and Harmony would happily erase a real pregnancy difference as if it were noise.

## 5. A first neural network: can the lipidome place a pixel in the brain?

Can we predict **where** a pixel sits (its in-plane CCF coordinates `yccf`, `zccf`) from its lipids alone? Position isn't a linear function of lipids, so we use a small **neural network** (a multilayer perceptron): stacked layers of weighted sums with a nonlinearity in between, trained to minimise prediction error. This is the atlas's "Lipid2Position" idea, shrunk to run on a CPU.

Two honesty tricks:
- **One hemisphere only.** The brain is left-right symmetric, so a pixel and its mirror twin have nearly identical lipids but opposite `zccf`. With both hemispheres the task would be impossible by construction.
- **Held-out pixels.** We train on 75% and score on the 25% the network never saw, with the per-axis Pearson r.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

z = adata.obs["zccf"].to_numpy()    # left-right coordinate
right = is_ctrl & (z >= np.median(z[is_ctrl]))            # control section, right hemisphere only
X_in = Xn[right]                                            # all lipids, 0-1 scaled
Y = adata.obs.loc[right, ["yccf", "zccf"]].to_numpy()    # the targets: in-plane position

tr, te = train_test_split(np.arange(right.sum()), test_size=0.25, random_state=SEED)    # 75% train / 25% held out
scaler = StandardScaler().fit(X_in[tr])                      # fit the scaling on TRAINING pixels only
# 3 hidden layers of 256, 128, 64 units with ReLU nonlinearities
net = MLPRegressor(hidden_layer_sizes=(256, 128, 64), activation="relu", max_iter=300,
                   early_stopping=True, random_state=SEED)   # early_stopping holds out 10% of train to stop in time
net.fit(scaler.transform(X_in[tr]), Y[tr])    # train
pred = net.predict(scaler.transform(X_in[te]))    # predict positions of pixels it never saw

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
for i, (ax, axis) in enumerate(zip(axes, ["yccf", "zccf"])):
    r = pearsonr(pred[:, i], Y[te][:, i])[0]    # agreement on this axis
    ax.scatter(Y[te][:, i], pred[:, i], s=2, alpha=0.2, color="mediumpurple", rasterized=True)
    lim = [Y[te][:, i].min(), Y[te][:, i].max()]    # ends of the perfect-prediction diagonal
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlabel(f"true {axis}"); ax.set_ylabel(f"predicted {axis}"); ax.set_aspect("equal")
    ax.set_title(f"{axis}: held-out r = {r:.2f}", fontsize=FS["m"])
    print(f"held-out Pearson r for {axis}: {r:.3f}")
fig.suptitle(f"lipids alone place a pixel in the brain (trained on {len(tr):,}, tested on {len(te):,} pixels)")
plt.tight_layout()
plt.show()

# Points hugging the diagonal mean the lipidome encodes location, not just texture.

## Save the embedded stage

In [ ]:
adata.var["moran_i"] = moran_ctrl.astype("float32")             # control section
adata.var["moran_i_pregnant"] = moran_preg.astype("float32")    # pregnant section
adata.var["selected"] = keep                                    # used for the embedding?
# a small record of the settings used
adata.uns["embedding"] = {"n_factors": N_FACTORS, "moran_threshold": MORAN_THR, "n_selected": int(keep.sum())}
adata.write_h5ad(paths.stage("embedded"))                       # -> data/derived/05_embedded.h5ad
print("saved", paths.stage("embedded"), "| obsm:", list(adata.obsm.keys()))

# ## Recap

- **Moran's I** measures spatial coherence; a permutation null shows 0.4 sits far above chance. We kept columns above 0.4 in both sections.
- **NMF** compresses them into non-negative, additive **programs**, learned on control and applied to both.
- **t-SNE** is for looking. It exposed a residual section split.
- **Harmony** mixes the sections in embedding space, for **clustering only**.
- A small neural net predicts a pixel's position from its lipids.
- Saved **`data/derived/05_embedded.h5ad`** with `X_nmf`, `X_tsne`, `X_harmony`.

**Next (06):** cluster the Harmonized embedding into lipizones and carry them onto the pregnant brain.